# Decision models as rerankers: the hosted walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hev/reranker/blob/main/notebooks/hosted_walkthrough.ipynb)

A decision model takes typed questions about a piece of text and returns calibrated probabilities. Asked *"is this
document relevant to the query?"*, it is a reranker. [`hev-rerank`](https://github.com/hev/reranker) calls five hosted
decision models in three request shapes with one function:

```python
rerank(query, docs, model="jev" | "perplexity" | "clef" | "clef-flash" | "openai", style="pair" | "batch" | "choice")
```

- **pair**: one call per (query, document), one yes/no probability each;
- **batch**: one call, the query and every candidate in one state, one yes/no question per document;
- **choice**: one call, one question *"which document is best?"*, probabilities that sum to one.

**Keys.** Each model needs its provider's key. Set whichever you have (Colab: the 🔑 *Secrets* panel, with
notebook access on; elsewhere: environment variables). Models without a key are skipped.

| model | variables |
|---|---|
| `jev` | `TYPESAFE_API_KEY` |
| `perplexity` | `PERPLEXITY_API_KEY` |
| `clef`, `clef-flash` | `CLOUDFLARE_API_TOKEN`, `CLOUDFLARE_ACCOUNT_ID` |
| `openai` | `OPENAI_API_KEY` |

No keys at all? [colab_free_tier.ipynb](https://colab.research.google.com/github/hev/reranker/blob/main/notebooks/colab_free_tier.ipynb)
runs an open-weight decision model on the free Colab GPU instead.

In [ ]:
# Parameters. HEV_RERANK_REF is the git tag or commit of hev-rerank to install.
HEV_RERANK_REF = "v0.3.0"
N_QUERIES = 30      # SciFact test queries per model and shape (pair makes K calls per query)
K = 30              # BM25 candidates per query

In [ ]:
%pip install -q "hev-rerank @ git+https://github.com/hev/reranker@{HEV_RERANK_REF}" rank_bm25

In [ ]:
import os
NEEDS = {"jev": ["TYPESAFE_API_KEY"], "perplexity": ["PERPLEXITY_API_KEY"],
         "clef": ["CLOUDFLARE_API_TOKEN", "CLOUDFLARE_ACCOUNT_ID"], "clef-flash": ["CLOUDFLARE_API_TOKEN", "CLOUDFLARE_ACCOUNT_ID"],
         "openai": ["OPENAI_API_KEY"]}
try:                                   # Colab secrets -> environment (values are never printed)
    from google.colab import userdata
    for var in {v for vs in NEEDS.values() for v in vs}:
        try:
            os.environ.setdefault(var, userdata.get(var))
        except Exception:
            pass
except ImportError:
    pass
MODELS = [m for m, vs in NEEDS.items() if all(os.environ.get(v) for v in vs)]
for m, vs in NEEDS.items():
    print(f"{m:11s} {'ready' if m in MODELS else 'skipped: set ' + ' and '.join(vs)}")

## Warm-up: six passages

In [ ]:
query = "does vitamin D supplementation improve bone density in older adults"
docs = [
 "Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone density in older adults.",
 "Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many older adults fall short.",
 "Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.",
 "Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineralization.",
 "Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults over 65 versus placebo.",
 "Astronauts lose 1 to 2 percent of bone density per month in microgravity.",
]

import hev_rerank
from hev_rerank import rerank, MODELS as TABLE
print(f"hev-rerank {hev_rerank.__version__}")
for m in MODELS:
    style = TABLE[m][2]                      # what style="auto" picks for this model
    print(f"\n{m} ({style}, its best measured shape):")
    for h in rerank(query, docs, model=m):
        print(f"  {h.score:6.3f}  [{h.index}] {docs[h.index][:90]}")
if not MODELS:
    print("No keys set: nothing to call. Set a key above, or use colab_free_tier.ipynb.")

The trial that answers the question and the meta-analysis that contradicts it should both come first: relevance
is *addresses the query*, not *agrees with it*. The background passage on calcium absorption is related but
doesn't answer it.

## SciFact: BM25, then every model in every shape

In [ ]:
import csv, io, json, math, random, re, urllib.request, zipfile
from rank_bm25 import BM25Okapi

# SciFact from BEIR (5,183 abstracts, 300 test queries with human relevance labels), 2.8 MB.
raw = urllib.request.urlopen("https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip").read()
z = zipfile.ZipFile(io.BytesIO(raw))
corpus = {d["_id"]: {"title": d["title"], "text": d["text"]} for d in map(json.loads, z.open("scifact/corpus.jsonl"))}
queries = {q["_id"]: q["text"] for q in map(json.loads, z.open("scifact/queries.jsonl"))}
qrels = {}
for row in csv.DictReader(io.TextIOWrapper(z.open("scifact/qrels/test.tsv")), delimiter="\t"):
    qrels.setdefault(row["query-id"], {})[row["corpus-id"]] = int(row["score"])

tok = lambda s: re.findall(r"\w+", s.lower())
ids = list(corpus)
bm25 = BM25Okapi([tok(corpus[i]["title"] + " " + corpus[i]["text"]) for i in ids])

def bm25_top(q, k=K):
    scores = bm25.get_scores(tok(q))
    return [ids[i] for i in sorted(range(len(ids)), key=lambda i: -scores[i])[:k]]

def ndcg10(ranked, rel):
    dcg = sum(rel.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked[:10]))
    ideal = sum(r / math.log2(i + 2) for i, r in enumerate(sorted(rel.values(), reverse=True)[:10]))
    return dcg / ideal if ideal else 0.0

qids = sorted(qrels)[:N_QUERIES]          # the first N test queries, sorted by id: the same subset every run
shortlists = {q: bm25_top(queries[q]) for q in qids}
bm25_ndcg = sum(ndcg10(shortlists[q], qrels[q]) for q in qids) / len(qids)
recall = sum(len(set(shortlists[q]) & set(qrels[q])) / len(qrels[q]) for q in qids) / len(qids)
print(f"{len(qids)} SciFact test queries, BM25 top-{K}: nDCG@10 {bm25_ndcg:.3f}, recall@{K} {recall:.3f} "
      "(a reranker can only reorder what BM25 found)")

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from hev_rerank import ProviderError

def pct(xs, p):
    xs = sorted(xs)
    return xs[min(len(xs) - 1, int(round(p / 100 * (len(xs) - 1))))]

def run(model, style):
    def one(q):
        t = time.perf_counter()
        hits = rerank(queries[q], [corpus[d] for d in shortlists[q]], model=model, style=style)
        return q, hits, time.perf_counter() - t
    with ThreadPoolExecutor(4) as ex:              # four queries in flight; pair fans out further inside rerank()
        return list(ex.map(one, qids))

results, scores = {}, {}
for m in MODELS:
    for style in ["pair", "batch", "choice"]:
        try:
            out = run(m, style)
        except ProviderError as e:                 # e.g. no beta access on this key: skip, keep going
            print(f"{m:11s} {style:6s} skipped: {str(e)[:120]}")
            continue
        nd = sum(ndcg10([shortlists[q][h.index] for h in hits], qrels[q]) for q, hits, _ in out) / len(out)
        lat = [s for _, _, s in out]
        results[m, style] = {"ndcg": nd, "p50": pct(lat, 50), "p95": pct(lat, 95)}
        scores[m, style] = {q: {shortlists[q][h.index]: h.score for h in hits} for q, hits, _ in out}
        print(f"{m:11s} {style:6s} nDCG@10 {bm25_ndcg:.3f} -> {nd:.3f}   p50 {pct(lat, 50):.2f} s  p95 {pct(lat, 95):.2f} s per query")

In [ ]:
from IPython.display import Markdown
rows = [f"| {m} | {s} | {r['ndcg']:.3f} | {r['ndcg'] - bm25_ndcg:+.3f} | {r['p50']:.2f} | {r['p95']:.2f} |"
        for (m, s), r in results.items()]
Markdown(f"{len(qids)} SciFact test queries, BM25 top-{K} (nDCG@10 {bm25_ndcg:.3f}). Latency per query, from wherever this ran.\n\n"
         "| model | shape | nDCG@10 | Δ vs BM25 | p50 s | p95 s |\n|---|---|---|---|---|---|\n" + "\n".join(rows))

With thirty queries, differences of a few points are within noise; the
[benchmark](https://hevmind.com/research/rerankers) uses 923 queries over three corpora and is where
`style="auto"` comes from.

## Calibration: do batch probabilities mean what they say?

A batch (or pair) score is a probability that the document is relevant. If it is calibrated, documents scored
around 0.8 should be relevant about 80% of the time. Bucket every (query, candidate) score and compare with the
human labels. (SciFact labels are sparse, about one relevant abstract per query, so the top buckets read low:
some "irrelevant" documents with high scores are unlabeled near-duplicates.)

In [ ]:
for m in MODELS:
    if (m, "batch") not in scores:
        continue
    pairs = [(s, d in qrels[q]) for q, ds in scores[m, "batch"].items() for d, s in ds.items()]
    print(f"\n{m}, batch: {len(pairs)} scored candidates")
    print("  score bucket   n     relevant")
    for lo, hi in [(0, .01), (.01, .1), (.1, .5), (.5, .9), (.9, 1.01)]:
        b = [r for s, r in pairs if lo <= s < hi]
        if b:
            print(f"  [{lo:.2f}, {min(hi, 1):.2f})  {len(b):5d}   {sum(b) / len(b):6.1%}")

## Choice can't be thresholded

Pair and batch scores are absolute, so they sort *and* prune: `threshold=0.1` drops candidates the model is
confident are irrelevant. Choice probabilities sum to one across the list; a document's choice score depends on
what else is in the list, so a fixed cut-off means nothing. `hev-rerank` raises instead of guessing, including
for `style="auto"` on models whose best shape is choice (Perplexity, OpenAI). This needs no key: it fails before
any call.

In [ ]:
try:
    rerank(query, docs, model="perplexity", style="choice", threshold=0.1)
except ValueError as e:
    print("ValueError:", e)
for m in MODELS:
    style = "batch" if TABLE[m][2] == "choice" else "auto"
    kept = rerank(query, docs, model=m, style=style, threshold=0.1)
    print(f"{m} {style}: keeps {[h.index for h in kept]} of {len(docs)} at threshold 0.1")

## Your own prompt

The question and its true/false criteria are plain text: change them to say what *relevant* means for your
corpus. `prompt=` overrides any of `question` (batch, with `{id}` for the document id), `pair_question`,
`choice_question` and `criteria`; anything you leave out keeps the default. Here, a stricter notion: only
**clinical trial evidence** counts.

In [ ]:
from hev_rerank import load_prompt
print("default batch question (Perplexity, Clef, OpenAI):", load_prompt(hev_rerank.local.PROMPT)["question"])
strict = {
    "question": "Does document `documents.{id}` report trial or meta-analysis evidence on the question in `query`?",
    "pair_question": "Does the document report trial or meta-analysis evidence on the question in the query?",
    "criteria": {"true": "The document reports results of a trial or a meta-analysis on the query's question, whatever the result.",
                 "false": "Background, mechanisms, dietary advice or other populations: not trial evidence on the question."},
}
for m in MODELS:
    style = "batch" if TABLE[m][2] == "choice" else TABLE[m][2]
    default = {h.index: h.score for h in rerank(query, docs, model=m, style=style)}
    custom = {h.index: h.score for h in rerank(query, docs, model=m, style=style, prompt=strict)}
    print(f"\n{m} ({style})   default -> trial-evidence prompt")
    for i in sorted(custom, key=lambda i: -custom[i]):
        print(f"  [{i}] {default[i]:6.3f} -> {custom[i]:6.3f}  {docs[i][:70]}")